# Deteksi Objek: Regresi Kotak, IoU, NMS, mAP, dan Detektor Pralatih

**Buku:** *Deep Learning dengan Python* — Bab 11, Segmentasi dan Deteksi Objek

Membangun intuisi deteksi dari bawah. Kita melatih model lokalisasi satu objek, menulis IoU, *non-maximum suppression*, dan *average precision* dengan NumPy, lalu menjalankan detektor Faster R-CNN pralatih.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±10 menit

**Isi notebook:**
- Bagian A (Keras): klasifikasi + regresi kotak pada data sintetis, dengan metrik IoU.
- Bagian B (NumPy): IoU, NMS, dan AP (interpolasi *all-point* ala PASCAL VOC), dicek dengan `torchvision.ops`.
- Bagian C: inferensi Faster R-CNN pralatih COCO dari `torchvision`. Dipilih karena API detektor pralatihnya paling stabil.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## Bagian A: lokalisasi satu objek
Setiap gambar berisi satu persegi atau elips. Model memprediksi **kelas** dan **kotak** $(x_1, y_1, x_2, y_2)$ ternormalisasi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras import layers, ops

def buat_data(n, ukuran=64, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.random((n, ukuran, ukuran, 3)).astype("float32") * 0.2
    kotak = np.zeros((n, 4), "float32"); kelas = rng.integers(0, 2, n).astype("float32")
    yy, xx = np.mgrid[:ukuran, :ukuran]
    for i in range(n):
        w, h = rng.integers(12, 40, size=2)
        x0, y0 = rng.integers(0, ukuran - w), rng.integers(0, ukuran - h)
        warna = rng.random(3) * 0.7 + 0.3
        if kelas[i] == 0:
            X[i, y0:y0 + h, x0:x0 + w] = warna                          # persegi
        else:
            cy, cx = y0 + h / 2, x0 + w / 2
            X[i][((yy - cy) / (h / 2)) ** 2 + ((xx - cx) / (w / 2)) ** 2 <= 1] = warna   # elips
        kotak[i] = [x0, y0, x0 + w, y0 + h]
    return X, kelas, kotak / ukuran

x_latih, k_latih, b_latih = buat_data(8000, seed=0)
x_uji, k_uji, b_uji = buat_data(1000, seed=1)

def gambar_kotak(ax, kotak, warna, ukuran=64, label=None):
    x1, y1, x2, y2 = np.asarray(kotak) * ukuran
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=warna, lw=2, label=label))

fig, ax = plt.subplots(1, 6, figsize=(14, 3))
for i, a in enumerate(ax):
    a.imshow(x_latih[i]); gambar_kotak(a, b_latih[i], "lime")
    a.set_title(["persegi", "elips"][int(k_latih[i])]); a.axis("off")
plt.show()

In [ ]:
def iou_kotak(y_benar, y_pred):
    x1 = ops.maximum(y_benar[:, 0], y_pred[:, 0]); y1 = ops.maximum(y_benar[:, 1], y_pred[:, 1])
    x2 = ops.minimum(y_benar[:, 2], y_pred[:, 2]); y2 = ops.minimum(y_benar[:, 3], y_pred[:, 3])
    irisan = ops.maximum(0.0, x2 - x1) * ops.maximum(0.0, y2 - y1)
    luas = lambda b: ops.maximum(0.0, b[:, 2] - b[:, 0]) * ops.maximum(0.0, b[:, 3] - b[:, 1])
    return irisan / (luas(y_benar) + luas(y_pred) - irisan + 1e-7)

masukan = keras.Input(shape=(64, 64, 3))
x = masukan
for f in [32, 64, 128]:
    x = layers.Conv2D(f, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
x = layers.Flatten()(x)                     # pertahankan informasi posisi (jangan global pooling)
x = layers.Dense(128, activation="relu")(x)
kelas = layers.Dense(1, activation="sigmoid", name="kelas")(x)
kotak = layers.Dense(4, activation="sigmoid", name="kotak")(x)
lokalisator = keras.Model(masukan, [kelas, kotak])

lokalisator.compile(optimizer="adam",
                    loss={"kelas": "binary_crossentropy", "kotak": keras.losses.Huber(delta=0.1)},
                    loss_weights={"kelas": 1.0, "kotak": 10.0},
                    metrics={"kelas": ["accuracy"], "kotak": [iou_kotak]})
lokalisator.fit(x_latih, {"kelas": k_latih, "kotak": b_latih}, epochs=15, batch_size=64, validation_split=0.1)

In [ ]:
p_kelas, p_kotak = lokalisator.predict(x_uji, verbose=0)
iou_uji = keras.ops.convert_to_numpy(iou_kotak(b_uji, p_kotak))
print(f"Akurasi kelas: {((p_kelas[:, 0] > 0.5) == k_uji).mean():.2%}")
print(f"IoU rata-rata: {iou_uji.mean():.3f} | IoU >= 0.5: {(iou_uji >= 0.5).mean():.2%}")

fig, ax = plt.subplots(1, 6, figsize=(14, 3))
for i, a in enumerate(ax):
    a.imshow(x_uji[i]); gambar_kotak(a, b_uji[i], "lime"); gambar_kotak(a, p_kotak[i], "red")
    a.set_title(f"IoU {iou_uji[i]:.2f}"); a.axis("off")
plt.suptitle("hijau = asli, merah = prediksi"); plt.show()

## Bagian B: IoU, NMS, dan *average precision* dengan NumPy

In [ ]:
def iou(kotak, daftar):
    """IoU satu kotak (4,) terhadap banyak kotak (N, 4); format (x1, y1, x2, y2)."""
    x1 = np.maximum(kotak[0], daftar[:, 0]); y1 = np.maximum(kotak[1], daftar[:, 1])
    x2 = np.minimum(kotak[2], daftar[:, 2]); y2 = np.minimum(kotak[3], daftar[:, 3])
    irisan = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    luas = lambda b: (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return irisan / (luas(kotak) + luas(daftar) - irisan)

def nms(kotak, skor, ambang=0.5):
    urutan = np.argsort(skor)[::-1]
    simpan = []
    while urutan.size:
        i = urutan[0]
        simpan.append(i)
        sisa = urutan[1:]
        urutan = sisa[iou(kotak[i], kotak[sisa]) < ambang]     # buang yang tumpang tindih
    return np.array(simpan)

def average_precision(kotak_det, skor, kotak_gt, ambang_iou=0.5):
    urutan = np.argsort(skor)[::-1]
    terpakai = np.zeros(len(kotak_gt), bool)
    tp = np.zeros(len(urutan))
    for r, i in enumerate(urutan):
        nilai = iou(kotak_det[i], kotak_gt)
        j = int(np.argmax(nilai))
        if nilai[j] >= ambang_iou and not terpakai[j]:
            tp[r], terpakai[j] = 1, True             # deteksi ganda untuk GT yang sama = FP
    tp_kum, fp_kum = np.cumsum(tp), np.cumsum(1 - tp)
    recall = tp_kum / len(kotak_gt)
    presisi = tp_kum / (tp_kum + fp_kum)
    r = np.concatenate([[0], recall, [1]]); p = np.concatenate([[0], presisi, [0]])
    for k in range(len(p) - 2, -1, -1):              # selubung presisi (monoton turun)
        p[k] = max(p[k], p[k + 1])
    idx = np.where(r[1:] != r[:-1])[0]
    return float(np.sum((r[idx + 1] - r[idx]) * p[idx + 1])), recall, presisi

In [ ]:
# Simulasi keluaran detektor mentah: banyak kotak berderau di sekitar 3 objek + beberapa salah deteksi
rng = np.random.default_rng(3)
gt = np.array([[20, 30, 120, 140], [150, 40, 260, 160], [80, 170, 200, 250]], float)
det, skor = [], []
for g in gt:
    for _ in range(6):
        det.append(g + rng.normal(0, 8, 4)); skor.append(rng.uniform(0.5, 0.99))
for _ in range(4):
    x, y = rng.uniform(0, 220, 2)
    det.append([x, y, x + rng.uniform(30, 80), y + rng.uniform(30, 80)]); skor.append(rng.uniform(0.3, 0.7))
det, skor = np.array(det), np.array(skor)

pertahankan = nms(det, skor, 0.5)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, idx, judul in [(ax[0], range(len(det)), f"Sebelum NMS ({len(det)} kotak)"),
                      (ax[1], pertahankan, f"Sesudah NMS ({len(pertahankan)} kotak)")]:
    a.set_xlim(0, 300); a.set_ylim(280, 0); a.set_title(judul)
    for g in gt:
        a.add_patch(plt.Rectangle(g[:2], g[2] - g[0], g[3] - g[1], fill=False, color="lime", lw=3))
    for i in idx:
        b = det[i]
        a.add_patch(plt.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, color="red", lw=1))
plt.show()

ap_mentah, _, _ = average_precision(det, skor, gt)
ap_nms, rec, pre = average_precision(det[pertahankan], skor[pertahankan], gt)
print(f"AP@0.5 tanpa NMS: {ap_mentah:.3f} | dengan NMS: {ap_nms:.3f}")

In [ ]:
# Cek implementasi kita terhadap pustaka resmi torchvision
import torch, torchvision
ref = torchvision.ops.nms(torch.tensor(det, dtype=torch.float32), torch.tensor(skor, dtype=torch.float32), 0.5)
print("NMS sama dengan torchvision:", sorted(ref.tolist()) == sorted(pertahankan.tolist()))
ref_iou = torchvision.ops.box_iou(torch.tensor(gt[:1]), torch.tensor(det)).numpy()[0]
print("IoU sama dengan torchvision:", np.allclose(ref_iou, iou(gt[0], det)))

## Bagian C: detektor pralatih (Faster R-CNN, COCO 80 kelas)

In [ ]:
from torchvision.io import read_image
from torchvision.utils import draw_bounding_boxes
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights

bobot = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
detektor = fasterrcnn_resnet50_fpn_v2(weights=bobot, box_score_thresh=0.5).eval()

jalur = keras.utils.get_file("coco_000000039769.jpg",
                             origin="http://images.cocodataset.org/val2017/000000039769.jpg")
gambar = read_image(jalur)                                    # uint8, (C, H, W)
with torch.no_grad():
    hasil = detektor([bobot.transforms()(gambar)])[0]

label = [f"{bobot.meta['categories'][k]} {s:.2f}" for k, s in zip(hasil["labels"], hasil["scores"])]
gambar_kotak_det = draw_bounding_boxes(gambar, hasil["boxes"], labels=label, colors="red", width=3)
plt.figure(figsize=(10, 7)); plt.imshow(gambar_kotak_det.permute(1, 2, 0)); plt.axis("off"); plt.show()
for l in label:
    print(l)

**Eksperimen:** turunkan `box_score_thresh` menjadi 0,1 dan amati deteksi tambahan yang muncul. Di aplikasi nyata, ambang ini dipilih dari kurva presisi–recall pada data validasi (Bab 16).